# Pattern 03: Hybrid retrieval (dense + BM25 via RRF)

Follows this repo's mandatory 8-section notebook template -- this is one of "the 10 patterns."

**This notebook's committed execution uses `RAG_RECIPES_LLM=mock`** for both the embedding and
generation steps. The BM25 half of this pattern is deterministic and needs no API call, but the
dense half's quality genuinely depends on the embedding model, so under `MockEmbedder` the fused
ranking below is illustrative of the code path only, not real retrieval quality. Section 7 is a
PENDING placeholder awaiting a real-embeddings run.


## Reproducibility header

In [1]:
import platform
import subprocess
import sys

import numpy
import openai

print(f"platform: {platform.platform()}")
print(f"python: {sys.version}")
print(f"openai sdk: {openai.__version__}")
print(f"numpy: {numpy.__version__}")

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="..").decode().strip()
except Exception:
    git_sha = "(not in a git repo checkout)"
print(f"git commit: {git_sha}")


platform: Windows-11-10.0.26200-SP0
python: 3.12.13 (main, Aug  7 2026, 02:26:41) [MSC v.1944 64 bit (AMD64)]
openai sdk: 2.53.0
numpy: 2.5.2
git commit: d8337d8fafa765c37169264abfbf530e3b1643e6


## Setup (loaded once, used by every section below)

In [2]:
import os

os.environ.setdefault("RAG_RECIPES_LLM", "mock")

from evals.run import load_corpus_by_id, load_qa_set, run_pattern
from recipes.llm import MockLLM, get_llm

corpus_by_id = load_corpus_by_id("../corpus/corpus.jsonl")
qa_set = load_qa_set("../evals/qa_set.jsonl")
llm = get_llm()  # used for generation (recipe_fn's own LLM calls)

# Judging needs its own backend: under a real API key this is the same
# real model, but under mock, `llm`'s canned generation text isn't valid
# JSON, and the judge prompts require JSON output. A separate MockLLM
# here demonstrates a clean, illustrative run instead of every question
# correctly (but noisily) failing to parse -- see evals/judges.py's
# JudgeParseError and evals/run.py's per-question error isolation.
if os.environ.get("RAG_RECIPES_LLM", "openai").lower() == "mock":
    judge_llm = MockLLM(default_response='{"score": 1, "reasoning": "Mock judge: looks fine."}')
else:
    judge_llm = llm

from recipes.embeddings import get_embedder

embedder = get_embedder()


## 1. What this pattern does

Hybrid retrieval runs dense (embedding-based) and BM25 (lexical) retrieval independently against
the same query, then fuses the two ranked lists with Reciprocal Rank Fusion (RRF):
`score(chunk) = sum over rankers of 1/(k + rank)`, with `k=60` (the standard literature default,
`recipes/hybrid.py`'s `RRF_K`). A chunk that ranks well in *either* list gets a high fused score;
a chunk found by both rankers scores higher still.


## 2. When to use it

- You don't know in advance whether queries will be keyword-heavy or paraphrase-heavy -- hybrid
  covers both failure modes at once (BM25's blind spot is dense's strength and vice versa)
- You want a retrieval upgrade over either pattern 01 or 02 alone without adding an LLM call
  (RRF is pure arithmetic, no extra API cost beyond the two retrieval calls themselves)
- Production systems where retrieval quality matters more than the ~2x retrieval latency/cost
  of running two rankers instead of one


## 3. When NOT to use it

- Retrieval latency budget can't absorb running two rankers (see pattern 01 or 02 alone)
- The corpus or query distribution strongly favors one ranker (e.g. pure keyword lookup over a
  code/API reference corpus) -- BM25 alone may already be near-optimal and hybrid adds cost with
  little upside
- You need to explain *why* a chunk was retrieved in a single, simple term -- an RRF-fused score
  is harder to justify to a user than either ranker's raw score alone


## 4. Implementation

In [3]:
from recipes.hybrid import make_retrieve_and_answer

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)

# Try it on one question directly.
sample = retrieve_and_answer("What does PCEval stand for?", k=3)
print("retrieved:", sample.retrieved_chunk_ids)
print("answer:", sample.answer)


retrieved: ['arxiv:2601.02404#0', 'arxiv:2601.00138#0', 'arxiv:2601.00138#2']
answer: PCEval stands for Physical Computing Evaluation. It is a benchmark designed for fully automatic evaluation of the capabilities of Large Language Models (LLMs) in both the logical and physical aspects of physical computing projects, without requiring human assessment [arxiv:2601.02404#0].


## 5. Run on our eval set

In [4]:
pattern_fn = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)

result = run_pattern(
    recipe_fn=pattern_fn,
    qa_set=qa_set,
    corpus_by_id=corpus_by_id,
    llm=judge_llm,
    pattern_name="03_hybrid_rrf",
    judges_enabled=True,
)


=== 03_hybrid_rrf (n=18) ===
  hit@3: 1.000  [95% CI 1.000, 1.000]
  hit@10: 1.000  [95% CI 1.000, 1.000]
  mrr: 0.889  [95% CI 0.778, 0.972]
  faithfulness: 0.667  [95% CI 0.444, 0.889]
  answer_relevance: 1.000  [95% CI 1.000, 1.000]
  citation_accuracy: 0.778  [95% CI 0.657, 0.880]
  filter_accuracy: 0.000  [95% CI 0.000, 0.000]
  p50_latency_ms: 3765.8
  p95_latency_ms: 35732.1
  usd_per_query: $0.01067
  eval_usd: $0.1921


## 6. Example query walkthrough

One example per eval-set category, showing the RRF-fused retrieved chunks and the (mocked) final
answer. Under `MockEmbedder`, the dense half's contribution is deterministic-but-arbitrary
(hash-based), so which chunks come back is not meaningful -- only that the fusion code path runs
end to end.

In [5]:
examples = {
    "keyword": "What does PCEval stand for?",
    "paraphrase": "Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?",
    "multi_hop": "The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?",
    "filter": "Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?",
}

for category, question in examples.items():
    result = retrieve_and_answer(question, k=3)
    print(f"--- {category} ---")
    print(f"Q: {question}")
    print(f"Retrieved: {result.retrieved_chunk_ids}")
    print(f"A: {result.answer}")
    print()


--- keyword ---
Q: What does PCEval stand for?
Retrieved: ['arxiv:2601.02404#0', 'arxiv:2601.00138#0', 'arxiv:2601.00138#2']
A: PCEval stands for Physical Computing Evaluation. It is a benchmark designed for fully automatic evaluation of the capabilities of Large Language Models (LLMs) in both the logical and physical aspects of physical computing projects, without requiring human assessment [arxiv:2601.02404#0].



--- paraphrase ---
Q: Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?
Retrieved: ['arxiv:2601.00090#0', 'arxiv:2601.00090#1', 'arxiv:2601.00090#2']
A: Repeated image generations from the same text prompt in diffusion models end up looking very similar due to a phenomenon called mode collapse. This occurs because the models tend to produce nearly identical outputs across different random seeds when given a fixed prompt, limiting the diversity of generated images. The initial noise inputs play a crucial role, but simply relying on random seeds often results in low variation. Thus, the model collapses to a few dominant modes, producing visually similar images repeatedly for the same prompt [arxiv:2601.00090#0, #1].



--- multi_hop ---
Q: The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?
Retrieved: ['arxiv:2601.00129#2', 'arxiv:2601.00129#0', 'arxiv:2601.00130#0']
A: The two photonics-AI-systems papers focus on complementary aspects of the challenge of realizing large-scale photonics-empowered AI systems:

1. The first paper, "Toward Large-Scale Photonics-Empowered AI Systems: From Physical Design Automation to System-Algorithm Co-Exploration" [arxiv:2601.00129], emphasizes the need for tightly coupled capabilities in both physical design automation and system-algorithm co-exploration. It highlights the challenges of scaling photonic integrated circuits (PICs) and electronic-photonic integrated circuits (EPICs) beyond isolated accelerators, addressing physical constraints such as routing, fabrication, thermal effects, and packaging. It also stresses the importance of incorporating physical non-idealities and control/c

--- filter ---
Q: Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?
Retrieved: ['arxiv:2601.00907#2', 'arxiv:2601.00907#0', 'arxiv:2601.00907#1']
A: The paper titled "Placenta Accreta Spectrum Detection using Multimodal Deep Learning" addresses diagnosing a pregnancy complication using deep learning. It focuses on Placenta Accreta Spectrum (PAS), a life-threatening obstetric condition, and develops a multimodal deep learning framework that integrates 3D MRI and 2D Ultrasound scans to improve early and accurate prenatal diagnosis. The model uses a 3D DenseNet121-Vision Transformer for MRI and a 2D ResNet50 for US, achieving superior diagnostic performance compared to unimodal models [arxiv:2601.00907#0, #1, #2].



## 7. Where this pattern FAILS

**PENDING: real findings from a one-off real-embeddings run.** A real `OPENAI_API_KEY` was not yet
available in the environment when this notebook was authored. This section will be replaced with a
static table of genuine hit@k/mrr failures (matching the format used in `02_bm25.ipynb`/
`04_rerank.ipynb` section 7), computed via a real, uncommitted exploratory run once a key is
available. The claim will be labeled with the date it was run
and its actual dollar cost, and will not be presented as live-executed cell output, to avoid
implying a mock re-run reproduces it (see this notebook's top-of-file disclaimer).


## 8. Copy-paste snippet

Meant for pasting into your own project, not executed as a cell in this notebook.

```python
"""Minimal hybrid (dense + BM25, RRF-fused) retrieval + generation, no eval harness."""
from recipes.embeddings import get_embedder
from recipes.hybrid import make_retrieve_and_answer
from recipes.llm import get_llm

corpus_by_id = {}  # {chunk_id: {"text": ..., ...}, ...} -- fill in your own chunks
embedder = get_embedder()
llm = get_llm()

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)
result = retrieve_and_answer("your question here", k=5)
print(result.answer)
```
